# NLP ESG KPI Extraction — Demo

Runs ingest → index → both extractors → comparison table → evaluation, end to end. All logic lives in `src/nlp_esg/`; this notebook is a thin driver.

In [1]:
from dotenv import load_dotenv
load_dotenv()

from nlp_esg.pipeline import (
    load_indexed_reports, run_extraction, load_gold_labels, run_evaluation,
)
from nlp_esg.compare import build_comparison_table

indexed = load_indexed_reports()
print(f'Loaded {len(indexed)} reports')

C:\Users\rotas\AppData\Roaming\Python\Python311\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Loaded 5 reports


In [2]:
extractions = run_extraction(indexed, include_llm=True)
print(f'Produced {len(extractions)} extractions')

Some weights of RobertaModel were not initialized from the model checkpoint at climatebert/distilroberta-base-climate-f and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
LLM call failed (attempt 1/3): Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CaSsRUUbEKHcp6GG8y1ym'}
LLM call failed (attempt 2/3): Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CaSsRZnL92rwPjoMq7smJ'}
LLM call failed (attempt 3/3): Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'me

Produced 30 extractions


## Comparison table — Baseline

In [3]:
build_comparison_table(extractions, extractor='baseline')

,scope_1_emissions,total_energy_consumption,water_consumption
bp,33700000.0,134448000.0,47300000.0
enel,18950000.0,168590000.0,32141000.0
eni,28400000.0,84399860.0,42000000.0
iberdrola,None,101572520.0,45642187.0
shell,None,269000000.0,72000000.0


## Comparison table — LLM

In [ ]:
build_comparison_table(extractions, extractor='llm')

## Evaluation — P / R / F1 / coverage

In [4]:
golds = load_gold_labels()
metrics = run_evaluation(extractions, golds)
metrics

,extractor,kpi,tp,fp,fn,tn,precision,recall,f1,coverage
0,baseline,scope_1_emissions,3,0,2,0,1.0,0.6,0.750000,0.6
1,baseline,total_energy_consumption,5,0,0,0,1.0,1.0,1.000000,1.0
2,baseline,water_consumption,4,1,0,0,0.8,1.0,0.888889,1.0
3,llm,scope_1_emissions,0,0,5,0,0.0,0.0,0.000000,0.0
4,llm,total_energy_consumption,0,0,5,0,0.0,0.0,0.000000,0.0
5,llm,water_consumption,0,0,5,0,0.0,0.0,0.000000,0.0


## Embedding-model comparison: MiniLM vs ClimateBERT

Re-run the baseline with MiniLM to compare retrieval quality.

In [ ]:
import os
os.environ['EMBEDDING_MODEL'] = 'minilm'
# Reload modules so the new env var takes effect. config caches EMBEDDING_MODEL_NAME
# at import time, so it must be reloaded before retrieval/pipeline.
import importlib, nlp_esg.config, nlp_esg.retrieval, nlp_esg.pipeline
importlib.reload(nlp_esg.config)
importlib.reload(nlp_esg.retrieval)
importlib.reload(nlp_esg.pipeline)
from nlp_esg.pipeline import load_indexed_reports as load2, run_extraction as run2, run_evaluation as eval2
indexed_mini = load2()
extractions_mini = run2(indexed_mini, include_llm=False)
eval2(extractions_mini, golds)

## Qualitative cases

Manually inspect cells where one extractor wins and the other doesn't. Fill in after running against real data.

- Baseline wins: Iberdrola total_energy (table path catches Total energy consumption (MWh) row that LLM-without-credits couldn't even attempt)
- LLM wins (when run with credits): Shell scope_1 (ESRS aggregation), Shell water (narrative reasoning) — per FINDINGS §6.14
- Both fail: Iberdrola scope_1 (gold value 5,246,890 lives on a row with no parseable label)